# Trial33 reduced swap trajectory

Explore the six-column reduced input in `data/raw/trial33`. This format is for visualization and is not accepted by the full swap-stat pipeline.

In [ ]:
import pathlib

import matplotlib.colors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repo_root() -> pathlib.Path:
    here = pathlib.Path.cwd().resolve()
    for base in (here, *here.parents):
        if (base / "pyproject.toml").is_file():
            return base
    raise FileNotFoundError(f"Could not locate the repository above {here}")


ROOT = find_repo_root()
DATA = ROOT / "data/raw/trial33/N40K-compress-LJ10-T0.001-F0.61-damp4.0_stage1_trial33_dt100-dr200__swaps-short.txt"
plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
swaps = pd.read_csv(DATA, sep="\t")
swaps.columns = [column.strip() for column in swaps.columns]
swaps["dt"] = swaps["Time"].diff()
swaps["cumulative_count"] = np.arange(1, len(swaps) + 1)
print(f"Events: {len(swaps):,}")
print(f"Time range: {swaps['Time'].min():.3f}–{swaps['Time'].max():.3f}")
swaps.describe()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0, 0].scatter(swaps["Time"], swaps["Snap direction"], s=5, alpha=0.4)
axes[0, 0].set(xlabel="time", ylabel="direction", title="Direction over time")
axes[0, 1].plot(swaps["Time"], swaps["cumulative_count"])
axes[0, 1].set(xlabel="time", ylabel="events", title="Cumulative snap number")
positive_dt = swaps.loc[swaps["dt"] > 0, "dt"]
axes[1, 0].hist(np.log10(positive_dt), bins=40)
axes[1, 0].set(xlabel="log10(dt)", title="Inter-event times")
normalization = matplotlib.colors.LogNorm(vmin=swaps["Time"].min(), vmax=swaps["Time"].max())
scatter = axes[1, 1].scatter(
    swaps["Snap bond c.x"], swaps["Snap bond c.y"], c=swaps["Time"],
    norm=normalization, s=4, alpha=0.6,
)
axes[1, 1].set(xlabel="x", ylabel="y", title="Bond centers colored by time")
axes[1, 1].set_aspect("equal", adjustable="box")
fig.colorbar(scatter, ax=axes[1, 1], label="time")
fig.tight_layout()
plt.show()